# LangChain create_agent를 A2A로 감싸기

이 노트북에서는 LangChain의 `create_agent`로 만든 에이전트를 A2A 프로토콜로 감싸는 방법을 학습합니다.

## 학습 목표
1. LangChain `create_agent`로 에이전트 생성
2. A2A `AgentCard` 생성
3. A2A 서버로 래핑하여 실행


In [28]:
# 필요한 패키지 설치 및 임포트
# %pip install --upgrade -q a2a-sdk langchain langchain-openai python-dotenv uvicorn nest-asyncio

import asyncio
import os
from dotenv import load_dotenv

load_dotenv()


True

## 1단계: LangChain create_agent로 에이전트 생성


In [32]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.graph.state import CompiledStateGraph

# 간단한 도구 예시
@tool
def search_tool(query: str) -> str:
    """검색 도구 - 주어진 쿼리에 대한 검색 결과를 반환합니다."""
    return f"'{query}'에 대한 검색 결과입니다. (예시 도구)"

# create_agent로 에이전트 생성
# create_agent는 CompiledStateGraph를 반환합니다
model = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

langchain_agent = create_agent(
    model=model,
    tools=[search_tool],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

print(f"✅ LangChain 에이전트 생성 완료!")
print(f"   타입: {type(langchain_agent)}")
print(f"   CompiledStateGraph 여부: {isinstance(langchain_agent, CompiledStateGraph)}")


✅ LangChain 에이전트 생성 완료!
   타입: <class 'langgraph.graph.state.CompiledStateGraph'>
   CompiledStateGraph 여부: True


In [33]:
from a2a.types import AgentCard, AgentCapabilities, AgentSkill, TransportProtocol

# Agent Card 생성
langchain_agent_card = AgentCard(
    name='LangChain Agent (A2A Wrapped)',
    url='http://localhost:10023',
    description='LangChain create_agent를 A2A로 래핑한 에이전트',
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='langchain_agent',
            name='LangChain Agent',
            description='LangChain create_agent로 만든 에이전트',
            tags=['langchain', 'a2a', 'agent'],
            examples=[
                'Hello, how can you help me?',
                'Search for information about AI',
            ],
        )
    ],
)

print(f"✅ Agent Card 생성 완료: {langchain_agent_card.name}")


✅ Agent Card 생성 완료: LangChain Agent (A2A Wrapped)


## 3단계: LangGraph를 A2A Executor로 래핑

LangGraph(CompiledStateGraph)를 A2A AgentExecutor로 래핑하는 클래스를 생성합니다.


In [ ]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.tasks import TaskUpdater
from a2a.types import TaskState, TextPart, Part
from a2a.utils import new_agent_text_message, new_task
from a2a.utils.errors import ServerError
from langchain_core.messages import HumanMessage, AIMessage

class LangChainA2AExecutor(AgentExecutor):
    """LangChain create_agent로 만든 에이전트를 A2A AgentExecutor로 래핑"""
    
    def __init__(self, graph: CompiledStateGraph):
        self.graph = graph
        # task.id 기준으로 실행 중인 asyncio.Task 관리
        self._running_tasks: dict[str, asyncio.Task] = {}
    
    async def _run_graph(
        self,
        task,
        updater: TaskUpdater,
        query: str,
    ) -> None:
        """LangGraph astream을 돌면서 A2A 이벤트로 스트리밍 전송"""
        config = {"configurable": {"thread_id": str(task.id)}}
        messages = [HumanMessage(content=query)] if query else []
        
        last_text = ""
        final_text = ""
        
        try:
            async for chunk in self.graph.astream({"messages": messages}, config=config):
                if isinstance(chunk, dict) and "messages" in chunk:
                    for msg in chunk["messages"]:
                        if isinstance(msg, AIMessage):
                            # content가 문자열인 경우
                            if isinstance(msg.content, str):
                                new_text = msg.content
                                if not new_text:
                                    continue
                                
                                # 전체 누적 텍스트 기준으로 delta 계산
                                if new_text == last_text:
                                    continue
                                
                                if new_text.startswith(last_text):
                                    delta = new_text[len(last_text):]
                                else:
                                    delta = new_text
                                
                                last_text = new_text
                                final_text = new_text  # 최종 텍스트 저장
                                
                                if delta:
                                    await updater.update_status(
                                        TaskState.working,
                                        new_agent_text_message(delta, task.context_id, task.id),
                                    )
                            # content가 리스트인 경우 (멀티모달)
                            elif isinstance(msg.content, list):
                                # 리스트에서 텍스트 추출
                                text_parts = []
                                for item in msg.content:
                                    if isinstance(item, dict) and 'text' in item:
                                        text_parts.append(item['text'])
                                    elif isinstance(item, str):
                                        text_parts.append(item)
                                
                                if text_parts:
                                    new_text = ''.join(text_parts)
                                    if new_text != last_text:
                                        if new_text.startswith(last_text):
                                            delta = new_text[len(last_text):]
                                        else:
                                            delta = new_text
                                        
                                        last_text = new_text
                                        final_text = new_text
                                        
                                        if delta:
                                            await updater.update_status(
                                                TaskState.working,
                                                new_agent_text_message(delta, task.context_id, task.id),
                                            )
            
            # 최종 결과 artifact 저장 (반드시 추가)
            if final_text:
                await updater.add_artifact([
                    Part(root=TextPart(text=final_text))
                ])
            elif last_text:
                # fallback: last_text 사용
                await updater.add_artifact([
                    Part(root=TextPart(text=last_text))
                ])
            
            # 완료 상태로 변경
            await updater.complete()
        
        except asyncio.CancelledError:
            raise
    
    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        query = context.get_user_input()
        
        # Task 확보
        task = context.current_task
        if not task:
            task = new_task(context.message)
            await event_queue.enqueue_event(task)
        
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        
        await updater.update_status(TaskState.submitted)
        
        try:
            await updater.start_work()
            
            # LangGraph 실행을 별도 Task로 관리 (cancel 지원)
            run_task = asyncio.create_task(
                self._run_graph(task, updater, query)
            )
            self._running_tasks[task.id] = run_task
            
            try:
                await run_task
            except asyncio.CancelledError:
                raise
        
        except asyncio.CancelledError:
            pass
        
        except Exception as e:
            error_msg = f"실행 중 오류 발생: {str(e)}"
            await updater.failed(
                message=new_agent_text_message(error_msg, task.context_id, task.id)
            )
            raise ServerError() from e
        
        finally:
            # 실행 끝난 Task 정리
            self._running_tasks.pop(task.id, None)
    
    async def cancel(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        """실행 중인 LangGraph Task 취소"""
        task = context.current_task
        if not task:
            return
        
        run_task = self._running_tasks.get(task.id)
        if run_task and not run_task.done():
            run_task.cancel()
        
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.failed(
            message=new_agent_text_message(
                "요청이 취소되었습니다.",
                task.context_id,
                task.id,
            )
        )

print("✅ LangChainA2AExecutor 클래스 정의 완료!")


✅ LangChainA2AExecutor 클래스 정의 완료!


## 4단계: A2A Executor로 래핑 및 서버 생성


In [37]:
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore


# A2A Executor로 래핑
langchain_a2a_executor = LangChainA2AExecutor(graph=langchain_agent)

# A2A 서버로 실행하기 위한 함수
def create_langchain_a2a_server(executor: LangChainA2AExecutor, agent_card: AgentCard):
    """LangChain 에이전트를 A2A 서버로 래핑"""
    request_handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),
    )
    
    return A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler
    )

# A2A 서버 애플리케이션 생성
a2a_server_app = create_langchain_a2a_server(langchain_a2a_executor, langchain_agent_card)

print("✅ A2A 서버 애플리케이션 생성 완료!")


✅ A2A 서버 애플리케이션 생성 완료!


## 5단계: A2A 서버 실행


In [ ]:
import uvicorn
import threading
import time
import nest_asyncio
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH

nest_asyncio.apply()

def run_a2a_server():
    """A2A 서버를 백그라운드에서 실행"""
    app = a2a_server_app.build()
    config = uvicorn.Config(
        app,
        host='127.0.0.1',
        port=10023,
        log_level='info',
    )
    server = uvicorn.Server(config)
    asyncio.run(server.serve())

# 백그라운드 스레드에서 서버 시작
server_thread = threading.Thread(target=run_a2a_server, daemon=True)
server_thread.start()

# 서버가 시작될 때까지 대기
time.sleep(3)

print("✅ A2A 서버 시작 완료!")
print(f"   URL: {langchain_agent_card.url}")
print(f"   Agent Card: {langchain_agent_card.url}{AGENT_CARD_WELL_KNOWN_PATH}")


INFO:     Started server process [22412]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:10023 (Press CTRL+C to quit)


✅ A2A 서버 시작 완료!
   URL: http://localhost:10023
   Agent Card: http://localhost:10023/.well-known/agent-card.json


INFO:     127.0.0.1:50302 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:50302 - "POST / HTTP/1.1" 200 OK


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO:     127.0.0.1:64684 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:64684 - "POST / HTTP/1.1" 200 OK


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## 6단계: A2A 클라이언트로 테스트


In [ ]:
import httpx
from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.types import AgentCard as A2AAgentCard
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH
from a2a.types import TextPart, DataPart, Message

class A2ASimpleClient:
    """A2A Simple 클라이언트 - 스트리밍 응답을 완전히 수집"""
    
    def __init__(self, default_timeout: float = 240.0):
        self.default_timeout = default_timeout
    
    def _extract_text_from_message(self, message: Message) -> str:
        """Message에서 텍스트 추출"""
        if not hasattr(message, 'parts') or not message.parts:
            return ""
        
        texts = []
        for part in message.parts:
            if hasattr(part, 'root'):
                root = part.root
                if isinstance(root, TextPart) and hasattr(root, 'text'):
                    texts.append(root.text)
                elif isinstance(root, DataPart) and hasattr(root, 'data'):
                    data = root.data
                    if isinstance(data, dict):
                        if 'text' in data:
                            texts.append(str(data['text']))
                        elif 'content' in data:
                            texts.append(str(data['content']))
        return ''.join(texts)
    
    async def create_task(self, agent_url: str, message: str) -> str:
        """A2A 에이전트에 메시지 전송 - 스트리밍 완전 수집"""
        timeout_config = httpx.Timeout(
            timeout=self.default_timeout,
            connect=10.0,
            read=self.default_timeout,
            write=10.0,
            pool=5.0,
        )
        
        async with httpx.AsyncClient(timeout=timeout_config) as httpx_client:
            # Agent Card 가져오기
            agent_card_response = await httpx_client.get(
                f'{agent_url}{AGENT_CARD_WELL_KNOWN_PATH}'
            )
            agent_card_data = agent_card_response.json()
            agent_card = A2AAgentCard(**agent_card_data)
            
            # A2A 클라이언트 생성
            config = ClientConfig(
                httpx_client=httpx_client,
                supported_transports=[
                    TransportProtocol.jsonrpc,
                    TransportProtocol.http_json,
                ],
                use_client_preference=True,
            )
            
            factory = ClientFactory(config)
            client = factory.create(agent_card)
            
            # 메시지 전송
            message_obj = create_text_message_object(content=message)
            
            # 스트리밍 응답 완전 수집
            final_task = None
            accumulated_text = ""  # 스트리밍 중 텍스트 누적
            artifact_texts = []    # 최종 artifact 텍스트
            
            async for response in client.send_message(message_obj):
                # response는 (task, message) 튜플
                if isinstance(response, tuple) and len(response) >= 2:
                    task = response[0]
                    msg = response[1] if len(response) > 1 else None
                    
                    final_task = task  # 최신 task로 업데이트
                    
                    # 1. 스트리밍 메시지에서 텍스트 추출 (진행 중 업데이트)
                    if msg and isinstance(msg, Message):
                        text = self._extract_text_from_message(msg)
                        if text:
                            # 중복 제거: 이미 포함된 텍스트는 제외
                            if text not in accumulated_text:
                                # 새로운 부분만 추가
                                if accumulated_text and text.startswith(accumulated_text):
                                    accumulated_text = text
                                else:
                                    accumulated_text += text
                    
                    # 2. Task의 artifacts에서 텍스트 추출 (최종 결과)
                    if hasattr(task, 'artifacts') and task.artifacts:
                        for artifact in task.artifacts:
                            if hasattr(artifact, 'parts') and artifact.parts:
                                for part in artifact.parts:
                                    if hasattr(part, 'root'):
                                        root = part.root
                                        if isinstance(root, TextPart) and hasattr(root, 'text'):
                                            artifact_texts.append(root.text)
                                        elif isinstance(root, DataPart) and hasattr(root, 'data'):
                                            data = root.data
                                            if isinstance(data, dict):
                                                if 'text' in data:
                                                    artifact_texts.append(str(data['text']))
                                                elif 'content' in data:
                                                    artifact_texts.append(str(data['content']))
            
            # 최종 결과 반환 (artifact 우선, 없으면 스트리밍 텍스트)
            if artifact_texts:
                return ''.join(artifact_texts)
            elif accumulated_text:
                return accumulated_text
            elif final_task:
                # 디버깅: task 상태 확인
                status = getattr(final_task, 'status', None)
                if status:
                    state = getattr(status, 'state', None)
                    print(f"⚠️ Task 상태: {state}, artifacts: {getattr(final_task, 'artifacts', None)}")
                return f"[응답 없음] Task ID: {getattr(final_task, 'id', 'unknown')}"
            else:
                return 'No response received'

# 클라이언트 생성 및 테스트
a2a_client = A2ASimpleClient()

async def test_agent():
    """에이전트 테스트"""
    response = await a2a_client.create_task(
        'http://localhost:10023',
        "Hello! Can you search for information about AI?"
    )
    print("📝 에이전트 응답:")
    print(response)

# 테스트 실행
asyncio.run(test_agent())


INFO:httpx:HTTP Request: GET http://localhost:10023/.well-known/agent-card.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:10023 "HTTP/1.1 200 OK"
INFO:a2a.client.client_task_manager:New task created with id: 5677a7b8-779a-4e4d-a59d-c2bb939cafff


📝 에이전트 응답:
artifacts=None context_id='cf4d395d-4913-4ed7-afe9-53cf42486d5f' history=[Message(context_id='cf4d395d-4913-4ed7-afe9-53cf42486d5f', extensions=None, kind='message', message_id='5cb9c017-ba40-4087-8716-ea827de33517', metadata=None, parts=[Part(root=TextPart(kind='text', metadata=None, text='Hello! Can you search for information about AI?'))], reference_task_ids=None, role=<Role.user: 'user'>, task_id='5677a7b8-779a-4e4d-a59d-c2bb939cafff')] id='5677a7b8-779a-4e4d-a59d-c2bb939cafff' kind='task' metadata=None status=TaskStatus(message=None, state=<TaskState.completed: 'completed'>, timestamp='2025-11-18T18:15:51.416525+00:00')


In [42]:
from langchain_openai import ChatOpenAI
# from langchain.agents import initialize_agent, Tool, AgentType
from langchain_classic.agents import initialize_agent, Tool, AgentType
from python_a2a.langchain import to_a2a_server
from python_a2a import run_server

# 1. LangChain 에이전트 생성
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0)

tools = [
    Tool(
        name="Calculator",
        func=lambda x: eval(x),
        description="Useful for math calculations"
    )
]

langchain_agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# 2. LangChain 에이전트를 A2A 서버로 변환
a2a_server = to_a2a_server(
    langchain_agent,
    name="Calculator Agent",
    description="An agent that can perform mathematical calculations"
)

TypeError: to_a2a_server() got an unexpected keyword argument 'name'

In [26]:
from langchain_openai import ChatOpenAI
# from langchain.agents import initialize_agent, Tool, AgentType
from langchain_classic.agents import initialize_agent, Tool, AgentType
from python_a2a.langchain import to_a2a_server
from python_a2a import run_server

# 1. LangChain 에이전트 생성
llm = ChatOpenAI(model= "gpt-4o-mini", temperature=0)

tools = [
    Tool(
        name="Calculator",
        func=lambda x: eval(x),
        description="Useful for math calculations"
    )
]

langchain_agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# 2. LangChain 에이전트를 A2A 서버로 변환
a2a_server = to_a2a_server(
    langchain_agent,
    name="Calculator Agent",
    description="An agent that can perform mathematical calculations"
)

TypeError: to_a2a_server() got an unexpected keyword argument 'name'

In [27]:
from a2a.types import AgentCard,  Message, TextPart
from a2a.server.agent_execution import AgentExecutor
from a2a.types import GetTaskRequest



# , TaskRequest,
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import uvicorn
from fastapi import FastAPI

# FastAPI 앱 생성
app = FastAPI()

# LangChain 에이전트 초기화
llm = ChatOpenAI(model="gpt-4")
tools = [...]  # your tools
agent = create_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

# Agent Card 생성
agent_card = AgentCard(
    name="My LangChain Agent",
    description="A LangChain agent exposed via A2A protocol",
    url="http://localhost:8000",
    version="1.0.0",
    capabilities={
        "streaming": False,
        "artifacts": True
    }
)

# Agent Card 엔드포인트
@app.get("/.well-known/agent.json")
async def get_agent_card():
    return agent_card.model_dump()

# Task 처리 엔드포인트
@app.post("/tasks")
async def create_task(task_request: GetTaskRequest):
    # Task에서 사용자 메시지 추출
    user_text = task_request.message.parts[0].text
    
    # LangChain 에이전트 실행
    result = agent_executor.invoke({"input": user_text})
    
    # A2A 응답 형식으로 반환
    return {
        "task_id": task_request.id,
        "status": "completed",
        "output": result["output"]
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)


NameError: name 'prompt' is not defined